# Trimodal vs Trimodal_curated Spider Plot

Radar comparison across grouped metrics overlaying trimodal and trimodal_curated (quilt1m_curated) models

In [ ]:
# Parameters from Snakemake
model_configs = snakemake.params.model_configs
metrics_by_modality = snakemake.params.metrics_by_modality


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
from pathlib import Path
from math import pi

# Apply matplotlib style
plt.style.use(snakemake.input.mpl_style)


In [ ]:
# Load data for both models (trimodal vs trimodal_curated)
model_data = {}

for i, (model_name, dataset_combo) in enumerate(model_configs):
    data = {}

    # HEST results - per-dataset performance
    with open(snakemake.input.hest_results[i]) as f:
        hest_data = json.load(f)
        for dataset_entry in hest_data.get("datasets", []):
            dataset_name = dataset_entry["dataset_name"]
            metric_key = "test_retrieval/transcriptome_image/rocauc_macroAvg"
            if metric_key in dataset_entry:
                data[f"hest_{dataset_name}"] = dataset_entry[metric_key]

    # MUSK results - balanced accuracy for pannuke and skin
    with open(snakemake.input.musk_results[i]) as f:
        musk_data = json.load(f)
        data["pannuke"] = musk_data["task_summaries"]["zeroshot_classification"][
            "balanced_acc"
        ]["pannuke"]
        data["skin"] = musk_data["task_summaries"]["zeroshot_classification"][
            "balanced_acc"
        ]["skin"]

    # CellWhisperer zero-shot and retrieval metrics
    cw_df = pd.read_csv(snakemake.input.cwevals_results[i], index_col=0)
    for metric in cw_df.index:
        data[metric] = cw_df.iloc[cw_df.index == metric, 0].values[0]

    model_data[model_name] = data


In [ ]:
# Prepare radar data
all_metrics = []
modality_labels = []
modality_colors = snakemake.params.modality_colors

for modality, metrics in metrics_by_modality.items():
    all_metrics.extend(metrics)
    modality_labels.extend([modality] * len(metrics))

radar_data = pd.DataFrame(index=all_metrics)
for model_name, data in model_data.items():
    radar_data[model_name] = [data.get(metric, 0) for metric in all_metrics]

# Normalize 0-1 per metric for visualization
radar_data_norm = radar_data.copy()
for metric in radar_data.index:
    max_val = radar_data.loc[metric].max()
    min_val = radar_data.loc[metric].min()
    if max_val > min_val:
        radar_data_norm.loc[metric] = (radar_data.loc[metric] - min_val) / (max_val - min_val)
    else:
        radar_data_norm.loc[metric] = 0.5


In [ ]:
# Plot radar for two models
N = len(all_metrics)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 5), subplot_kw=dict(projection="polar"))

model_colors = [
    modality_colors["trimodal"],
    '#555555'  # curated overlay color
]

for i, (model_name, color) in enumerate(zip(radar_data.columns, model_colors)):
    values = radar_data[model_name].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

# Metric labels
ax.set_xticks(angles[:-1])
metric_labels = [metric.replace('valfn_zshot_TabSap_', '').replace('/', '\n') for metric in all_metrics]
ax.set_xticklabels(metric_labels, fontsize=10)

# Color labels by modality
for angle, label, modality in zip(angles[:-1], ax.get_xticklabels(), modality_labels):
    label.set_color(modality_colors[modality])
    label.set_fontweight('bold')

rad_max = float(np.nanmax(radar_data.values))
ax.set_ylim(0, rad_max if rad_max > 0 else 1.0)
tick_vals = np.linspace(0, ax.get_ylim()[1], 5)
ax.set_yticks(tick_vals)
ax.set_yticklabels([f"{v:.2f}" for v in tick_vals], fontsize=8)
# default polar radial ticks

ax.grid(True, alpha=0.3)

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=color, label=model) for model, color in zip(radar_data.columns, model_colors)]
ax.legend(handles=legend_handles, loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=12, title='Models')

plt.title('Trimodal vs Trimodal_curated Performance Across Modality Pairings', size=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(snakemake.output.plot, dpi=300, bbox_inches='tight')
plt.savefig(snakemake.output.plot_svg, dpi=300, bbox_inches='tight')
plt.show()
